# HW-2: полный инференс в Google Colab

Ноутбук скачивает `test.parquet`, генерирует решения для всех 1919 задач в исходном порядке и сохраняет `predictions.jsonl`. Рекомендуется заранее положить в одну папку Google Drive четыре файла: `inference-best.pt`, `source_tokenizer.json`, `target_tokenizer.json`, а также исходники `model.py` и `inference.py`.

In [ ]:
%pip install -q tokenizers pyarrow pandas

## Подключение файлов

По умолчанию используется папка `MyDrive/hw2-code-generation`. Измените `DRIVE_ARTIFACT_DIR`, если положили файлы в другое место. Если Google Drive не нужен, установите `USE_GOOGLE_DRIVE = False`: Colab предложит выбрать пять файлов вручную.

In [ ]:
from pathlib import Path

USE_GOOGLE_DRIVE = True
DRIVE_ARTIFACT_DIR = Path('/content/drive/MyDrive/hw2-code-generation')

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    ARTIFACT_DIR = DRIVE_ARTIFACT_DIR
else:
    from google.colab import files
    print('Выберите checkpoint, оба tokenizer JSON, model.py и inference.py')
    files.upload()
    ARTIFACT_DIR = Path('/content')

required = ['source_tokenizer.json', 'target_tokenizer.json', 'model.py', 'inference.py']
missing = [name for name in required if not (ARTIFACT_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f'В {ARTIFACT_DIR} отсутствуют файлы: {missing}')
print('Файлы модели:', ARTIFACT_DIR)

In [ ]:
import gc
import json
import sys
import time
import urllib.request

import pandas as pd
import torch
from tokenizers import Tokenizer

OUTPUT_PATH = Path('/content/predictions.jsonl')
QUESTIONS_PATH = Path('/content/test.parquet')
TEST_URL = 'https://huggingface.co/datasets/SlavaYus/tinysms_qa_12m/resolve/main/data/test/test.parquet?download=true'
EXPECTED_COUNT = 1919
BATCH_SIZE = 32
MAX_NEW_TOKENS = 256
NUM_BEAMS = 3
LENGTH_PENALTY = 1.0

if not torch.cuda.is_available():
    raise RuntimeError('В Colab выберите Runtime → Change runtime type → GPU')
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
checkpoint_candidates = [
    ARTIFACT_DIR / 'inference-best.pt',
    ARTIFACT_DIR / 'inference.pt',
    ARTIFACT_DIR / 'best.pt',
    ARTIFACT_DIR / 'model.pt',
]
checkpoint_path = next((path for path in checkpoint_candidates if path.is_file()), None)
if checkpoint_path is None:
    raise FileNotFoundError(f'В {ARTIFACT_DIR} не найден checkpoint')

if not QUESTIONS_PATH.exists():
    urllib.request.urlretrieve(TEST_URL, QUESTIONS_PATH)

sys.path.insert(0, str(ARTIFACT_DIR))
from inference import generate_texts, load_model

test = pd.read_parquet(QUESTIONS_PATH)
prompt_column = next(
    (name for name in ('question', 'prompt', 'text') if name in test.columns),
    None,
)
if prompt_column is None:
    raise KeyError(f'Не найдена колонка с условием; колонки: {list(test.columns)}')
prompts = test[prompt_column].astype(str).tolist()
if len(prompts) != EXPECTED_COUNT:
    raise ValueError(f'Ожидалось {EXPECTED_COUNT} задач, получено {len(prompts)}')

print('Checkpoint:', checkpoint_path)
print('Загружено задач:', len(prompts))

In [ ]:
device = torch.device('cuda')
source_tokenizer = Tokenizer.from_file(str(ARTIFACT_DIR / 'source_tokenizer.json'))
target_tokenizer = Tokenizer.from_file(str(ARTIFACT_DIR / 'target_tokenizer.json'))
model = load_model(checkpoint_path, device)
print(f'Параметров: {sum(parameter.numel() for parameter in model.parameters()):,}')
print('max_source_length:', model.config.max_source_length)
print('max_new_tokens:', MAX_NEW_TOKENS)
print('num_beams:', NUM_BEAMS)

In [ ]:
started = time.perf_counter()
batch_size = BATCH_SIZE

while True:
    try:
        predictions = generate_texts(
            model,
            source_tokenizer,
            target_tokenizer,
            prompts,
            batch_size=batch_size,
            max_source_length=model.config.max_source_length,
            max_new_tokens=MAX_NEW_TOKENS,
            num_beams=NUM_BEAMS,
            length_penalty=LENGTH_PENALTY,
        )
        break
    except torch.cuda.OutOfMemoryError:
        if batch_size == 1:
            raise
        batch_size = max(1, batch_size // 2)
        gc.collect()
        torch.cuda.empty_cache()
        print('Недостаточно GPU-памяти; повторяем с batch_size =', batch_size)

elapsed = time.perf_counter() - started
print(f'Готово за {elapsed / 60:.1f} мин.; batch_size={batch_size}')

In [ ]:
if len(predictions) != EXPECTED_COUNT:
    raise ValueError(f'Ожидалось {EXPECTED_COUNT} предсказаний, получено {len(predictions)}')

with OUTPUT_PATH.open('w', encoding='utf-8') as file:
    for prediction in predictions:
        file.write(json.dumps({'code': prediction}, ensure_ascii=False) + '\n')

with OUTPUT_PATH.open(encoding='utf-8') as file:
    result = [json.loads(line) for line in file]

assert len(result) == EXPECTED_COUNT
assert all(set(row) == {'code'} and isinstance(row['code'], str) for row in result)
print('Сохранено:', OUTPUT_PATH)
print('Строк:', len(result))
result[:3]

In [ ]:
from google.colab import files
files.download(str(OUTPUT_PATH))